# 7 · Reaction handling

## ZHS

| `zhs` | |
|---|---|
| `kinetic` (default) | finite-rate precipitation and dissolution from the local supersaturation |
| `equilibrium` | always at equilibrium where present |
| `lumped` | formed in proportion to R1 and R2 so they consume no net H⁺ (Bernard et al. 2024) |
| `off` | none |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {z: run(Params(zhs=z, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for z in ("kinetic", "equilibrium", "lumped", "off")}
summary(runs)
compare(runs, "ZHS treatment", cols=("V", "pH_cath", "vf_ZHS"));

## The insertion OCP

The same material reacts by insertion (R3) and by dissolution (R2): a discharge goes through one and then the other, and a charge has to undo both. With the empirical OCP alone (`spline`) the host fills *exactly* and then cannot release Zn, so a charge recovers only what R2 deposition supplies. Nernstian ends (`spline_nernst`, the default) or a Nernstian OCP (`nernst`) keep both directions open (`docs/options.md`).

In [ ]:
runs = {o: run(Params(r3_ocp=o, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for o in ("spline_nernst", "spline", "nernst")}
summary(runs)
compare(runs, "insertion OCP", cols=("V", "theta"));

## Switching reactions off

In [ ]:
runs = {
    "all reactions": run(Params(steps="cc I=100 Vmin=1.0")),
    "R1 off": run(Params(R1_on=False, steps="cc I=100 Vmin=1.0")),
    "R2 off": run(Params(R2_on=False, steps="cc I=100 Vmin=1.0")),
    "R3 off": run(Params(R3_on=False, steps="cc I=100 Vmin=1.0")),
}
summary(runs)
compare(runs, "reaction switches");

## Nucleation threshold for ZHS

In [ ]:
runs = {f"zhs_nucleation = {s}": run(Params(zhs_nucleation=s, steps="cc I=100 Vmin=1.0")) for s in (1.0, 1.05, 1.2)}
summary(runs)
compare(runs, "ZHS nucleation threshold", cols=("V", "pH_cath", "vf_ZHS"));